# Transfer Learning y Fine Tuning con CNN

## 0. Importación de librerías

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix

## 1. Preparación del dataset

In [2]:
DATA_DIR = 'data'

train_folders = [
    'github_train_0',
    'github_train_1',
    'github_train_2',
    'github_train_3'
]

test_folder = 'github_test'

### Función para crear DataFrame con rutas y etiquetas

In [3]:
def build_dataframe(folder_path):
    images = []
    labels = []

    for file in os.listdir(folder_path):
        if file.lower().endswith(('jpg','jpeg','png')):
            label = file.split('_')[0]
            images.append(os.path.join(folder_path, file))
            labels.append(label)

    return pd.DataFrame({
        'filename': images,
        'class': labels
    })

In [4]:
train_df_list = []

for folder in train_folders:
    path = os.path.join(DATA_DIR, folder)
    df = build_dataframe(path)
    train_df_list.append(df)

train_df = pd.concat(train_df_list, ignore_index=True)

test_df = build_dataframe(os.path.join(DATA_DIR, test_folder))

print('Train size:', len(train_df))
print('Test size:', len(test_df))
train_df.head()

Train size: 4000
Test size: 1000


,filename,class
0,data\github_train_0\cat.1000.jpg,cat.1000.jpg
1,data\github_train_0\cat.10010.jpg,cat.10010.jpg
2,data\github_train_0\cat.10012.jpg,cat.10012.jpg
3,data\github_train_0\cat.10013.jpg,cat.10013.jpg
4,data\github_train_0\cat.10017.jpg,cat.10017.jpg


## 2. Generadores de datos

In [5]:
IMG_SIZE = (224,224)
BATCH_SIZE = 32

In [6]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [7]:
train_generator = datagen.flow_from_dataframe(
    train_df,
    x_col='filename',
    y_col='class',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

Found 3200 validated image filenames belonging to 4000 classes.


In [8]:
val_generator = datagen.flow_from_dataframe(
    train_df,
    x_col='filename',
    y_col='class',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 800 validated image filenames belonging to 4000 classes.


In [9]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col='filename',
    y_col='class',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 1000 validated image filenames belonging to 1000 classes.


## 3. Modelo preentrenado MobileNetV2

In [10]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

## 4. Transfer Learning

In [11]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)

num_classes = train_df['class'].nunique()

predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

In [12]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 33s 313ms/step - accuracy: 0.0000e+00 - loss: 8.3194 - val_accuracy: 0.0000e+00 - val_loss: 8.3375
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 19s 193ms/step - accuracy: 0.0000e+00 - loss: 8.2625 - val_accuracy: 0.0000e+00 - val_loss: 8.7157
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 19s 193ms/step - accuracy: 9.3750e-04 - loss: 7.9821 - val_accuracy: 0.0000e+00 - val_loss: 9.3469
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 19s 188ms/step - accuracy: 0.0078 - loss: 6.9884 - val_accuracy: 0.0000e+00 - val_loss: 10.5834
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 20s 201ms/step - accuracy: 0.0834 - loss: 5.1851 - val_accuracy: 0.0000e+00 - val_loss: 13.0162


## 5. Fine Tuning

In [14]:
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [15]:
history_ft = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 26s 234ms/step - accuracy: 0.2466 - loss: 3.8646 - val_accuracy: 0.0000e+00 - val_loss: 13.9346
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 236ms/step - accuracy: 0.4106 - loss: 2.8786 - val_accuracy: 0.0000e+00 - val_loss: 14.7899
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 237ms/step - accuracy: 0.5350 - loss: 2.3184 - val_accuracy: 0.0000e+00 - val_loss: 15.0739
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 26s 259ms/step - accuracy: 0.6600 - loss: 1.8026 - val_accuracy: 0.0000e+00 - val_loss: 15.7383
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 23s 232ms/step - accuracy: 0.7500 - loss: 1.4021 - val_accuracy: 0.0000e+00 - val_loss: 16.8362


### Evaluación Fine Tuning